In [2]:
import pandas as pd
import networkx as nx
from matplotlib import pyplot as plt
import numpy as np

In [43]:
edge_list = pd.read_csv(r"D:\commo\code\4_kumu_struct\edge_list.csv")
node1 = pd.read_csv(r"D:\commo\code\4_kumu_struct\node_list1.csv")
node2 = pd.read_csv(r"D:\commo\code\4_kumu_struct\node_list2.csv")
node3 = pd.read_csv(r"D:\commo\code\4_kumu_struct\node_list3.csv")
others = pd.read_csv(r"D:\commo\code\4_kumu_struct\others.csv")

In [4]:
all_documented_labels = set(node1['label']) | set(node2['label']) | set(node3['label']) | set(others['label'])
edge_labels = set(edge_list['From']) | set(edge_list['To'])

undocumented = edge_labels - all_documented_labels
print(len(undocumented), "undocumented entities out of", len(edge_labels), "total in edge_list")
print(sorted(undocumented))

0 undocumented entities out of 163 total in edge_list
[]


In [ ]:
# pyG

In [ ]:
# create a mapping of unique node1 'label' indices from range [0, num_node1]
# create a mapping of unique node2 'label' indices from range [0, num_node2]
# create a mapping of unique node3 'label' indices from range [0, num_node3]
# create a mapping of unique others 'label' indices from range [0, num_others]


# filter out edge_list where Type = 'owns'
# create a mapping of unique edge_list 'From' indices from range [0, num_From_owns] (look up node1, node2, node3, others for existing mappedID)
# create a mapping of unique edge_list 'To' indices from range [0, num_To_owns] (look up node1, node2, node3, others for existing mappedID)

# filter out edge_list where Type = 'counsels_for'
# create a mapping of unique edge_list 'From' indices from range [0, num_From_counsels] (look up node1, node2, node3, others for existing mappedID)
# create a mapping of unique edge_list 'To' indices from range [0, num_To_counsels] (look up node1, node2, node3, others for existing mappedID)

# filter out edge_list where Type = 'lends_to'
# create a mapping of unique edge_list 'From' indices from range [0, num_From_lends] (look up node1, node2, node3, others for existing mappedID)
# create a mapping of unique edge_list 'To' indices from range [0, num_To_lends] (look up node1, node2, node3, others for existing mappedID)

In [44]:
# create a mapping of unique node1 'label' indices from range [0, num_node1]
unique_node1_label = node1['label'].unique()
unique_node1_label = pd.DataFrame(data={
    'label': unique_node1_label,
    'mappedID': pd.RangeIndex(len(unique_node1_label)),
})
print("Mapping of node1 labels to consecutive values:")
print("==========================================")
print(unique_node1_label.head())
print()

# create a mapping of unique node2 'label' indices from range [0, num_node2]
unique_node2_label = node2['label'].unique()
unique_node2_label = pd.DataFrame(data={
    'label': unique_node2_label,
    'mappedID':pd.RangeIndex(len(unique_node2_label))
})
print('Mapping of node2 labels to consecutive values:')
print("==========================================")
print(unique_node2_label.head())
print()

# create a mapping of unique node3 'label' indices from range [0, num_node3]
unique_node3_label = node3['label'].unique()
unique_node3_label = pd.DataFrame(data={
    'label': unique_node3_label,
    'mappedID':pd.RangeIndex(len(unique_node3_label))
})
print('Mapping of node3 labels to consecutive values:')
print("==========================================")
print(unique_node3_label.head())
print()

# create a mapping of unique others 'label' indices from range [0, num_others]
unique_others_label = others['label'].unique()
unique_others_label = pd.DataFrame(data={
    'label': unique_others_label,
    'mappedID':pd.RangeIndex(len(unique_others_label))
})
print('Mapping of others labels to consecutive values:')
print("==========================================")
print(unique_others_label.head())
print()

Mapping of node1 labels to consecutive values:
                        label  mappedID
0             trafigura group         0
1                  vitol asia         1
2           mercuria holdings         2
3            gunvor singapore         3
4  louis dreyfus company asia         4

Mapping of node2 labels to consecutive values:
                       label  mappedID
0                 reed smith         0
1  holman fenwick willan hfw         1
2                 clyde & co         2
3         stephenson harwood         3
4   watson farley & williams         4

Mapping of node3 labels to consecutive values:
                label  mappedID
0                 ing         0
1    societe generale         1
2         bnp paribas         2
3            rabobank         3
4  standard chartered         4

Mapping of others labels to consecutive values:
                                    label  mappedID
0                                  amaroq         0
1    argoglobal underwriting asia paci

In [62]:
# combined lookup
combined_lookup = pd.concat(
    [
        unique_node1_label.set_index("label")["mappedID"],
        unique_node2_label.set_index("label")["mappedID"],
        unique_node3_label.set_index("label")["mappedID"],
        unique_others_label.set_index("label")["mappedID"],
    ]
)
# check duplicates in combine_lookup
#combined_lookup[combined_lookup.index.duplicated(keep=False)]

In [83]:
# filter out edge_list where Type = 'owns'
owns_edge_list = edge_list[edge_list['Type']=='owns']

# create a mapping of unique edge_list 'From' indices from range [0, num_From_owns] (look up node1, node2, node3, others for existing mappedID)
own_from_unique = owns_edge_list["From"].unique()
own_from_unique = pd.DataFrame(data={'from': own_from_unique})
own_from_unique['mappedID'] = own_from_unique['from'].map(combined_lookup)

# create a mapping of unique edge_list 'To' indices from range [0, num_To_owns] (look up node1, node2, node3, others for existing mappedID)
own_to_unique = owns_edge_list["To"].unique()
own_to_unique = pd.DataFrame(data={"to": own_to_unique})
own_to_unique["mappedID"] = own_to_unique["to"].map(combined_lookup)

# Perform merge to obtain the 'owns' edges from 'from' and 'to':
owns_from_id = pd.merge(owns_edge_list['From'], own_from_unique, left_on='From', right_on='from', how='left')
owns_to_id = pd.merge(owns_edge_list['To'], own_to_unique, left_on='To', right_on='to', how='left')

# convert mapped IDs to tensors
import torch
owns_from_id = torch.from_numpy(owns_from_id["mappedID"].values)
owns_to_id = torch.from_numpy(owns_to_id["mappedID"].values)

# construct 'own_edge_index' in COO format
owns_edge_index_from_to = torch.stack([owns_from_id, owns_to_id], dim=0)
assert owns_edge_index_from_to.size() == (2,26) # 26 rows in owns_edge_list
print()
print("Final edge indices pointing From To with edge type 'owns':")
print("=================================================")
print(owns_edge_index_from_to)


Final edge indices pointing From To with edge type 'owns':
tensor([[ 40,   0,   0, 134, 128, 128, 125,   6,   6, 126, 123, 124,  13,  69,
         132, 132, 132, 132,  56, 114,   4,  42,   5, 122, 121,  11],
        [  0,  32,  41,   1, 129,   2,   3, 112,  24,   4,   7,   9, 130,  15,
          57,  58, 133,  52, 131,  20,  72,  67,  27,   8,  11,  68]])


In [95]:
# filter out edge_list where Type = 'counsels_for'
counsels_edge_list = edge_list[edge_list["Type"] == "counsels_for"]
counsels_edge_list # 78 rows

# create a mapping of unique edge_list 'From' indices from range [0, num_From_counsels] (look up node1, node2, node3, others for existing mappedID)
counsels_from_unique = counsels_edge_list["From"].unique()
counsels_edge_list

,From,To,Direction,Type
34,reed smith,china merchants energy shipping holding singapore,directed,counsels_for
35,reed smith,sinopec,directed,counsels_for
36,reed smith,trafigura maritime logistics,directed,counsels_for
37,reed smith,uniper global commodities se singapore branch,directed,counsels_for
38,holman fenwick willan hfw,bgn energy,directed,counsels_for
...,...,...,...,...
107,incisive law llc,china taiping insurance singapore,directed,counsels_for
108,incisive law llc,great american insurance company,directed,counsels_for
109,incisive law llc,ms first capital insurance,directed,counsels_for
110,incisive law llc,qbe insurance singapore,directed,counsels_for


In [ ]:
# stellarGraph's ComplEx, PyG's models